In [2]:
import sys
import os

# 1. Direct Python to the root project folder
sys.path.append(os.path.abspath(os.path.join('..')))

In [3]:
from dotenv import load_dotenv
load_dotenv()


True

In [4]:
from openai import OpenAI
openai_client = OpenAI()

In [6]:
from scripts.ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [7]:
def search(query):
    boost_dict = {'question':3.0}
    
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description":"Search query text to look up in the FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties":False
    }
}

In [9]:
messages = [
    {'role': 'user', 'content': 'How can i creat an account?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"create an account signup register account how can I create an account"}', call_id='call_3n9s870IHZn3wIXFn0JD7LMT', name='search', type='function_call', id='fc_077d9d6f5b6fc178006a6d37a66ca4819badc38aba67404d8d', caller=None, namespace=None, status='completed')]

In [10]:
call = response.output[0]

In [11]:
call.arguments

'{"query":"create an account signup register account how can I create an account"}'

In [12]:
import json

args = json.loads(call.arguments)

search_results = search(**args)

result_json = json.dumps(search_results, indent=2)

In [13]:
print(messages)
messages.extend(response.output)

print(messages)

[{'role': 'user', 'content': 'How can i creat an account?'}]
[{'role': 'user', 'content': 'How can i creat an account?'}, ResponseFunctionToolCall(arguments='{"query":"create an account signup register account how can I create an account"}', call_id='call_3n9s870IHZn3wIXFn0JD7LMT', name='search', type='function_call', id='fc_077d9d6f5b6fc178006a6d37a66ca4819badc38aba67404d8d', caller=None, namespace=None, status='completed')]


In [14]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

In [15]:
messages.append(function_call_output)

In [16]:
messages

[{'role': 'user', 'content': 'How can i creat an account?'},
 ResponseFunctionToolCall(arguments='{"query":"create an account signup register account how can I create an account"}', call_id='call_3n9s870IHZn3wIXFn0JD7LMT', name='search', type='function_call', id='fc_077d9d6f5b6fc178006a6d37a66ca4819badc38aba67404d8d', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_3n9s870IHZn3wIXFn0JD7LMT',
  'output': '[\n  {\n    "question": "How can I create an account?",\n    "answer": "To create an account, click on the \'Sign Up\' button on the top right corner of our website and follow the instructions to complete the registration process.",\n    "id": "HN6d6WSY"\n  },\n  {\n    "question": "Can I order without creating an account?",\n    "answer": "Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.",\n    "id": "uFdvvamx"\n  },

In [17]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [18]:

print(response.output_text)

To create an account, click the **“Sign Up”** button in the **top right corner** of the website and follow the registration steps.

If you want, I can also help with what information you’ll need to sign up.


In [19]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(431, 52)

In [20]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [21]:
instructions = """
You're a Ecommerce Chatbot assistant.
You're given a question from a customer and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other query that the user wants to get answer.
""".strip()

In [22]:
def make_call(call):
    args = json.loads(call.arguments)

    result = ''
    if call.name == "search":
        result = search(**args)
    
    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json
    }

In [23]:
question = "How can i create my account?"

messages = [
    {"role": "developer", "content": instructions},
    {"role":"user", "content": question}
]

response = openai_client.responses.create(
    model = "gpt-5.4-mini",
    input = messages,
    tools = [search_tool]
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True
    
    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"create account sign up register account"}
function_call: search {"query":"how can I create my account"}
function_call: search {"query":"account creation sign up login registration"}


In [24]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:

    messages = [
        {"role": "developer", "content": instructions},
        {"role":"user", "content": question}
    ]

    last_answer = ''

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model = model,
            input = messages,
            tools = [search_tool]
        )

        usage = response.usage
        print(f'Tokens({usage.input_tokens}, {usage.output_tokens})')
        result = calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)
        print("Total cost: $", round(result["total_cost"], 8))

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True
            
            elif item.type == "message":
                last_answer = item.content[0].text
        
        it = it + 1
        if has_function_calls == False:
            break
    return last_answer

In [28]:
instructions = """
You're a ecommerce chatbot assistant.
You're given a question from a customer and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the ecommerce site, orders, products or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other questions that the user wants to be answered.
""".strip()

In [29]:
question = "What's qeen gambit"

answer = agent_loop(instructions, question)
print(answer)

iteration #1...
Tokens(217, 20)
Total cost: $ 4.455e-05
function_call: search {"query":"qeen gambit"}
iteration #2...
Tokens(248, 47)
Total cost: $ 6.54e-05
Sorry, I can’t help with that here.

If you have a question about the ecommerce site, products, orders, or delivery, I’m happy to help. Do you have any other questions I can answer?


In [30]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [31]:
def search(query: str) -> dict[str,str]:
    """
    Search the FAQ database for entries matching the given query.
    """

    return index.search(
        query,
        num_results=5,
        boost_dict={'question':3.0}
    )

In [32]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [33]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [34]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [35]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [37]:
result = runner.loop(
    prompt='How do i register for my account?',
    callback=callback
)

-> Response received


-> Response received


In [38]:
result.cost

CostInfo(input_cost=Decimal('0.000597'), output_cost=Decimal('0.0003195'), total_cost=Decimal('0.0009165'))